# INSPECT AUGMENTATION OUTPUT

In [ ]:
from __future__ import annotations

from pathlib import Path
import numpy as np

from emma import AugmentedDataset, EmmaDataset
from emma.augment import HDF5Source
import hdf5plugin

from functools import cache
from textwrap import shorten

## > DATASET

In [ ]:
DATASET_ROOT = Path("PATH/TO/EMMa")
AUGMENTATION_PATH = Path("PATH/TO/CLIP-H5-FILE")
SOURCE_NAME = "clip_vit_b32_category_hierarchy_embeddings"

if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(
        f"Please set DATASET_ROOT to a valid EMMa dataset directory. Got: {DATASET_ROOT}"
    )

if not AUGMENTATION_PATH.is_file():
    raise FileNotFoundError(
        f"Please set AUGMENTATION_PATH to a valid augmentation file. Got: {AUGMENTATION_PATH}"
    )

In [ ]:
ds = EmmaDataset(DATASET_ROOT)

In [ ]:
source = HDF5Source(SOURCE_NAME, AUGMENTATION_PATH, mode="r")

In [ ]:
ads = AugmentedDataset(ds).register(source)

## > INSPECT

In [ ]:
SAMPLE_SIZE = 1_000
N_EXAMPLES = 6
SEED = 7

rng = np.random.default_rng(SEED)
augmented_asins = list(ads.source(SOURCE_NAME).keys())
sample_asins = rng.choice(
    augmented_asins,
    min(SAMPLE_SIZE, len(augmented_asins)),
    replace=False,
).tolist()

In [ ]:
batch = ads.source(SOURCE_NAME).get_batch(sample_asins)
X_raw = np.stack([batch[asin] for asin in sample_asins])
X = X_raw.astype(np.float64)

if X.ndim != 2:
    raise ValueError(f"Expected vectors, got sample shape {X.shape}")

print(f"Embeddings: {len(source):,} | pool: {len(augmented_asins):,} | sampled: {len(X):,}")
print(f"Shape: {X.shape} | stored dtype: {X_raw.dtype}")

In [ ]:
def quantiles(values):
    qs = np.quantile(values, [0, .05, .25, .5, .75, .95, 1])
    return "  ".join(f"{name}={value:.5g}" for name, value in zip(
        ("min", "p05", "p25", "p50", "p75", "p95", "max"), qs
    ))

finite = np.isfinite(X).all(axis=1)
norms = np.linalg.norm(X, axis=1)
near_zero = norms < 1e-8
unique_vectors = np.unique(X, axis=0).shape[0]
coordinate_std = X.std(axis=0)

print(f"Finite vectors:           {finite.sum():,}/{len(X):,}")
print(f"Near-zero vectors:        {near_zero.sum():,}/{len(X):,}")
print(f"Exact unique vectors:     {unique_vectors:,}/{len(X):,}")
print(f"Non-varying coordinates:  {(coordinate_std < 1e-12).sum():,}/{X.shape[1]:,}")
print(f"Mean absolute coordinate: {np.abs(X).mean():.6g}")
print(f"Coordinate std:           {X.std():.6g}")
print("Norms:", quantiles(norms))

if not finite.all():
    raise ValueError("Sample contains NaN or infinite values.")
if near_zero.any():
    print("WARNING: near-zero vectors found.")
if unique_vectors == 1:
    print("WARNING: all sampled embeddings are identical.")
elif unique_vectors < len(X):
    print("Note: exact duplicates exist.")

In [ ]:
X_unit = X / np.maximum(norms[:, None], np.finfo(float).eps)
cosine = np.clip(X_unit @ X_unit.T, -1, 1)
pair_cosines = cosine[np.triu_indices(len(X), k=1)]

print(f"Pairwise comparisons: {len(pair_cosines):,}")
print("Cosines:", quantiles(pair_cosines))
print(f"Mean ± std: {pair_cosines.mean():.5f} ± {pair_cosines.std():.5f}")
print(f"Pairs > 0.99: {(pair_cosines > .99).mean():.2%}")

if pair_cosines.std() < 1e-3:
    print("WARNING: cosine similarities have almost no variation.")
if np.quantile(pair_cosines, .05) > .99:
    print("WARNING: embeddings may be collapsed.")

s = np.linalg.svd(X - X.mean(0), full_matrices=False, compute_uv=False) ** 2
explained = s / s.sum()
effective_rank = np.exp(-(explained[explained > 0] * np.log(explained[explained > 0])).sum())

print(f"Top-1 centered variance:  {explained[0]:.2%}")
print(f"Top-10 centered variance: {explained[:10].sum():.2%}")
print(f"Effective rank:           {effective_rank:.1f} / {min(X.shape)}")

In [ ]:
@cache
def metadata(asin):
    try:
        rec = ads.get_record(asin)
        return ads.get_title(rec=rec), ads.get_category_names(rec=rec)
    except (KeyError, IndexError, TypeError, ValueError):
        return "", []

def label(asin):
    title, categories = metadata(asin)
    text = " > ".join(map(str, categories)) or str(title) or "(metadata unavailable)"
    return shorten(text, width=100, placeholder="…")

sample_asins_array = np.asarray(sample_asins)

def show_neighbors(asin, scores, n_near=3, n_far=2):
    candidates = np.flatnonzero(sample_asins_array != asin)
    ranked = candidates[np.argsort(scores[candidates])]
    print(f"\nQUERY {asin:>14}  {label(asin)}")
    for rank, i in enumerate(ranked[-n_near:][::-1], 1):
        print(f"NEAR {rank:<2} cos={scores[i]: .4f}  {sample_asins[i]:>14}  {label(sample_asins[i])}")
    for rank, i in enumerate(ranked[:n_far], 1):
        print(f"FAR  {rank:<2} cos={scores[i]: .4f}  {sample_asins[i]:>14}  {label(sample_asins[i])}")

In [ ]:
for i in rng.choice(len(X), min(N_EXAMPLES, len(X)), replace=False):
    show_neighbors(sample_asins[i], cosine[i])

In [ ]:
pairs = [
    (i, metadata(asin)[1][0])
    for i, asin in enumerate(sample_asins[:300])
    if metadata(asin)[1]
]

if len(pairs) < 2:
    print("Skipped: fewer than two sampled products had category metadata.")
else:
    indices, categories = zip(*pairs)
    local = cosine[np.ix_(indices, indices)]
    upper = np.triu_indices(len(indices), k=1)
    scores = local[upper]
    categories = np.asarray(categories, dtype=object)
    same = categories[upper[0]] == categories[upper[1]]
    within, across = scores[same], scores[~same]

    print(f"Products with metadata: {len(indices):,}")
    print(f"Same-category pairs:     {len(within):,}")
    print(f"Cross-category pairs:    {len(across):,}")
    print(f"Same-category cosine:    median={np.median(within):.4f}, mean={within.mean():.4f}")
    print(f"Cross-category cosine:   median={np.median(across):.4f}, mean={across.mean():.4f}")
    print(f"Median separation:       {np.median(within) - np.median(across):+.4f}")

In [ ]:
QUERY_ASIN = None
query_asin = QUERY_ASIN or sample_asins[0]

query = np.asarray(
    ads.get_augmentation(SOURCE_NAME, query_asin),
    dtype=np.float64,
)
query_norm = np.linalg.norm(query)
query_scores = X_unit @ (query / max(query_norm, np.finfo(float).eps))

print(f"Query norm: {query_norm:.5g}")
show_neighbors(query_asin, query_scores, n_near=10, n_far=5)